# Load Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import pandas as pd
import anndata as ad
import spatialdata as sd
from scipy.stats import median_abs_deviation
from sklearn.cluster import KMeans
from spatialdata_io import xenium_explorer_selection
import os
import re
import sopa
from scipy.spatial import cKDTree
import matplotlib.ticker as ticker

sc.set_figure_params(dpi=200)
sc.logging.print_header()

In [ ]:
SCLE_1 = sd.read_zarr("/Users/minhu/irAE/Processed_Data/SCLE_1.zarr")
SCLE_2 = sd.read_zarr("/Users/minhu/irAE/Processed_Data/SCLE_2.zarr")

SCLE_3 = sd.read_zarr("/Users/minhu/irAE1/Processed_Data/SCLE_3.zarr")
SCLE_4 = sd.read_zarr("/Users/minhu/irAE1/Processed_Data/SCLE_4.zarr")
SCLE_5 = sd.read_zarr("/Users/minhu/irAE1/Processed_Data/SCLE_5.zarr")

NS1 = sd.read_zarr("/Users/minhu/irAE1/Processed_Data/Normal_1.zarr")
NS2 = sd.read_zarr("/Users/minhu/irAE1/Processed_Data/Normal_2.zarr")
NS3 = sd.read_zarr("/Users/minhu/irAE1/Processed_Data/Normal_3.zarr")
NS4 = sd.read_zarr("/Users/minhu/irAE1/Processed_Data/Normal_4.zarr")

In [ ]:
SCLE_1.tables['table'].obs['Sample'] = 'irAE-SCLE1'
SCLE_2.tables['table'].obs['Sample'] = 'irAE-SCLE2'
SCLE_3.tables['table'].obs['Sample'] = 'SCLE1'
SCLE_4.tables['table'].obs['Sample'] = 'SCLE2'
SCLE_5.tables['table'].obs['Sample'] = 'SCLE3'
NS1.tables['table'].obs['Sample'] = 'NS1'
NS2.tables['table'].obs['Sample'] = 'NS2'
NS3.tables['table'].obs['Sample'] = 'NS3'
NS4.tables['table'].obs['Sample'] = 'NS4'

SCLE_1.tables['table'].obs['Batch'] = 'Batch1'
SCLE_2.tables['table'].obs['Batch'] = 'Batch1'
SCLE_3.tables['table'].obs['Batch'] = 'Batch1'
SCLE_4.tables['table'].obs['Batch'] = 'Batch2'
SCLE_5.tables['table'].obs['Batch'] = 'Batch2'
NS1.tables['table'].obs['Batch'] = 'Batch2'
NS2.tables['table'].obs['Batch'] = 'Batch2'
NS3.tables['table'].obs['Batch'] = 'Batch2'
NS4.tables['table'].obs['Batch'] = 'Batch2'

SCLE_1.tables['table'].obs['Lesion'] = 'SCLE'
SCLE_2.tables['table'].obs['Lesion'] = 'SCLE'
SCLE_3.tables['table'].obs['Lesion'] = 'SCLE'
SCLE_4.tables['table'].obs['Lesion'] = 'SCLE'
SCLE_5.tables['table'].obs['Lesion'] = 'SCLE'
NS1.tables['table'].obs['Lesion'] = 'NS'
NS2.tables['table'].obs['Lesion'] = 'NS'
NS3.tables['table'].obs['Lesion'] = 'NS'
NS4.tables['table'].obs['Lesion'] = 'NS'

SCLE_1.tables['table'].obs['Lesion1'] = 'irAE-SCLE'
SCLE_2.tables['table'].obs['Lesion1'] = 'irAE-SCLE'
SCLE_3.tables['table'].obs['Lesion1'] = 'SCLE'
SCLE_4.tables['table'].obs['Lesion1'] = 'SCLE'
SCLE_5.tables['table'].obs['Lesion1'] = 'SCLE'
NS1.tables['table'].obs['Lesion1'] = 'NS'
NS2.tables['table'].obs['Lesion1'] = 'NS'
NS3.tables['table'].obs['Lesion1'] = 'NS'
NS4.tables['table'].obs['Lesion1'] = 'NS'

SCLE_1.tables['table'].obs['Patient'] = 'Patient1'
SCLE_2.tables['table'].obs['Patient'] = 'Patient1'
SCLE_3.tables['table'].obs['Patient'] = 'Patient2'
SCLE_4.tables['table'].obs['Patient'] = 'Patient3'
SCLE_5.tables['table'].obs['Patient'] = 'Patient3'
NS1.tables['table'].obs['Patient'] = 'Patient4'
NS2.tables['table'].obs['Patient'] = 'Patient5'
NS3.tables['table'].obs['Patient'] = 'Patient6'
NS4.tables['table'].obs['Patient'] = 'Patient7'

In [ ]:
adata = ad.concat([SCLE_1.tables["table"],SCLE_2.tables["table"],SCLE_3.tables["table"],SCLE_4.tables["table"],
                   SCLE_5.tables["table"],NS1.tables["table"],NS2.tables["table"],NS3.tables["table"],
                   NS4.tables["table"]],join='inner',uns_merge='unique')
adata.obs_names = adata.obs['Sample'] + '_' + adata.obs['cell_id']
adata

In [ ]:
sc.pp.calculate_qc_metrics(adata, inplace=True)
adata.obs['nucleus_ratio'] = adata.obs["nucleus_area"] / adata.obs["cell_area"]

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))

axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)


axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata.obs["cell_area"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)

In [ ]:
samples = sorted(adata.obs["Sample"].unique())
metrics = [
    ("total_counts", "Total transcripts per cell"),
    ("n_genes_by_counts", "Unique transcripts per cell"),
    ("cell_area", "Area of segmented cells"),
    ("nucleus_ratio", "Nucleus ratio"),
]

# shared x-limits and bins per metric
xlims = {}
bins_dict = {}

for col, _ in metrics:
    vals = adata.obs[col].dropna().values
    xmin, xmax = np.percentile(vals, [0.5, 99.5])  # robust to outliers
    xlims[col] = (xmin, xmax)
    bins_dict[col] = np.linspace(xmin, xmax, 50)   # same bins across rows

ymax = {col: 0 for col, _ in metrics}

fig, axs = plt.subplots(
    len(samples), 
    len(metrics), 
    figsize=(4 * len(metrics), 3 * len(samples)),
    sharex=False, sharey=False
)

for i, sample in enumerate(samples):
    sample_adata = adata[adata.obs["Sample"] == sample]

    for j, (col, title) in enumerate(metrics):
        ax = axs[i, j] if len(samples) > 1 else axs[j]

        sns.histplot(
            sample_adata.obs[col],
            bins=bins_dict[col],
            kde=False,
            ax=ax,
        )

        ax.set_xlim(xlims[col])
        ax.grid(False)

        if i == 0:
            ax.set_title(title, fontsize=18)

        ax.set_xlabel("")
        ax.set_ylabel("")

        # track max y across rows for this column
        ymax[col] = max(ymax[col], ax.get_ylim()[1])

    axs[i, 0].set_ylabel(sample, fontsize=18)

for j, (col, _) in enumerate(metrics):
    for i in range(len(samples)):
        axs[i, j].set_ylim(0, ymax[col])

plt.tight_layout(rect=[0, 0.08, 1, 1])
# plt.savefig("/Users/minhu/LM/Plots/LM_QC.pdf", transparent=False)

In [ ]:
print(adata.obs[['nucleus_area', 'cell_area']].isna().sum())

In [ ]:
adata = adata[adata.obs['nucleus_area']>0].copy()

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.filter_cells(adata,min_genes=1)
sc.pp.filter_genes(adata,min_cells=1)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.pca(adata, random_state=42)
sc.pp.neighbors(adata, random_state=42)
sc.tl.umap(adata,random_state=42)

In [ ]:
sc.pl.violin(adata, ['log1p_n_genes_by_counts', 'log1p_total_counts','nucleus_ratio'],stripplot=False, multi_panel=True)

In [ ]:
sc.pl.umap(adata,color=['Sample',"segmentation_method",'log1p_n_genes_by_counts', 'log1p_total_counts',
                        'n_genes_by_counts', 'total_counts','cell_area',
                        'nucleus_ratio',],ncols=2)

In [ ]:
adata.obs['segmentation_method'].value_counts()

In [ ]:
adata = adata[adata.obs['segmentation_method']!='Segmented by nucleus expansion of 5.0µm'].copy()
adata

In [ ]:
sc.pl.violin(adata, ['log1p_n_genes_by_counts', 'log1p_total_counts','nucleus_ratio',"n_genes_by_counts","total_counts"],stripplot=False, multi_panel=True)

In [ ]:
sc.pp.pca(adata, random_state=42)
sc.pp.neighbors(adata, random_state=42)
sc.tl.umap(adata,random_state=42)

In [ ]:
sc.pl.umap(adata,color=['Sample',"segmentation_method",'log1p_n_genes_by_counts', 'log1p_total_counts'],ncols=2)

In [ ]:
def flag_outliers_by_mad(adata, column, upper_mad=5, lower_mad=5):
    values = adata.obs[column].values
    med = np.median(values)
    mad_val = median_abs_deviation(values)
    
    if upper_mad != 0:
        high_cut = med + upper_mad * mad_val
    else: high_cut = max(values)

    if lower_mad != 0:
        low_cut  = med - lower_mad * mad_val
    else: low_cut = min(values)
    
    column_name = column + "_outlier"
    adata.obs[column_name] = (values > high_cut) | (values < low_cut)

    print(column, "Upper MAD: ", high_cut)
    print(column, "Lower MAD: ", low_cut)
    return adata

In [ ]:
adata = flag_outliers_by_mad(adata, 'log1p_total_counts', upper_mad=0, lower_mad=2)
adata = flag_outliers_by_mad(adata, 'log1p_n_genes_by_counts', upper_mad=0, lower_mad=2)

In [ ]:
adata.obs['log1p_total_counts_outlier'].value_counts()

In [ ]:
adata.obs['log1p_n_genes_by_counts_outlier'].value_counts()

In [ ]:
outlier_columns = [
    'log1p_total_counts_outlier',
    'log1p_n_genes_by_counts_outlier',
]

adata.obs['Outlier'] = adata.obs[outlier_columns].any(axis=1)

In [ ]:
sc.pl.umap(adata,color=['Outlier'])

# Filtered

In [ ]:
adata_filtered = adata[adata.obs['Outlier']==False].copy()
adata_filtered

In [ ]:
sc.pl.violin(adata_filtered, ['log1p_n_genes_by_counts', 'log1p_total_counts',"n_genes_by_counts","total_counts",'nucleus_ratio'],stripplot=False, multi_panel=True)

In [ ]:
sc.pp.pca(adata_filtered,random_state=42)
sc.pp.neighbors(adata_filtered,random_state=42)
sc.tl.umap(adata_filtered,random_state=42)

In [ ]:
sc.pl.umap(adata_filtered,color=['log1p_n_genes_by_counts', 'log1p_total_counts',
                        'n_genes_by_counts', 'total_counts','cell_area',
                        'nucleus_ratio'])

In [ ]:
sc.pl.umap(adata_filtered, color=["Sample","Batch","Lesion"])

# Cell Type Annotation

In [ ]:
sc.pl.umap(adata_filtered,color=["CDSN","COL17A1","DSG1","MITF","DCT","MLANA",
                                 "PDGFRA","SCN7A",
                                 "CD3G","CD4",'CD19','MS4A1','CLEC10A','CLEC9A',"CD14",'LAMP3','CD207','CLEC4C','XBP1',"CSF3R",
                                 "CD68"])

In [ ]:
sc.tl.leiden(adata_filtered, resolution=0.2, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(adata_filtered,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(adata_filtered,groupby='leiden')
sc.tl.dendrogram(adata_filtered,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(adata_filtered,groupby='leiden',show_gene_labels=True,swap_axes=True)

In [ ]:
Cell_Type = {'0': 'Immune Cells',
 '1': 'Fibroblasts',
 '2': 'Adipocytes',
 '3': 'ECs',
 '4': 'Eccrine Gland Cells',
 '5': 'Mast Cells',
 '6': 'Keratinocytes',
 '7': 'Keratinocytes',
 '8': 'Keratinocytes',
 "9": "Eccrine Ductal Cells"
}
adata_filtered.obs['Cell_Type'] = adata_filtered.obs['leiden'].map(Cell_Type)
sc.pl.umap(adata_filtered, color=["Cell_Type"],legend_loc='on data',legend_fontsize=5,legend_fontoutline=1)

## Endothelial

In [ ]:
endo = adata_filtered[adata_filtered.obs["Cell_Type"]=="ECs"].copy()

sc.pp.pca(endo,random_state=42)
sc.pp.neighbors(endo,random_state=42)
sc.tl.umap(endo,random_state=42)

In [ ]:
sc.pl.umap(endo,color=[
                       "RGS5","NOTCH3","ABCC9",
                       "MYLK",
                       "PECAM1","CD34","PLVAP",
                       "FLT4","PDPN","PROX1"])

In [ ]:
sc.pl.umap(endo,color=['Sample',"Lesion","Lesion1"])

In [ ]:
sc.tl.leiden(endo, resolution=0.2, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(endo,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(endo,groupby='leiden')
sc.tl.dendrogram(endo,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(endo,groupby='leiden',show_gene_labels=True,swap_axes=True, n_genes=20)

In [ ]:
Cell_Type = {'0': 'Pericytes',
 '1': 'vECs',
 '2': 'lECs',
 '3': 'vSMCs',
}

endo.obs['Cell_Type1'] = endo.obs['leiden'].map(Cell_Type)

In [ ]:
endo_map = endo.obs['Cell_Type1']
overlapping_indices = adata_filtered.obs.index.intersection(endo_map.index)

adata_filtered.obs['Cell_Type1'] = adata_filtered.obs['Cell_Type'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type1'] = endo_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color="Cell_Type1")

In [ ]:
ec = endo[endo.obs["Cell_Type1"]=="vECs"].copy()

sc.pp.pca(ec,random_state=42)
sc.pp.neighbors(ec,random_state=42, n_neighbors=10, n_pcs=15)
sc.tl.umap(ec,random_state=42)

In [ ]:
sc.pl.umap(ec, color=["Sample","Lesion","Lesion1"])

In [ ]:
sc.tl.leiden(ec, resolution=0.3, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(ec,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(ec,groupby='leiden')
sc.tl.dendrogram(ec,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(ec,groupby='leiden',show_gene_labels=True,swap_axes=True, n_genes=15)

In [ ]:
Cell_Type2 = {'0': 'vECs',
              '1': 'Activated vECs',
              '2': 'IFN vECs',
              '3': 'vECs',
}
ec.obs['Cell_Type2'] = ec.obs['leiden'].map(Cell_Type2)

sc.pl.umap(ec,color="Cell_Type2")

In [ ]:
ec_scle = ec[ec.obs["Cell_Type2"]=="IFN vECs"].copy()

sc.pp.pca(ec_scle,random_state=42)
sc.pp.neighbors(ec_scle,random_state=42, n_neighbors=10, n_pcs=15)
sc.tl.umap(ec_scle,random_state=42)

In [ ]:
sc.tl.leiden(ec_scle, resolution=0.3, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(ec_scle,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(ec_scle,groupby='leiden')
sc.tl.dendrogram(ec_scle,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(ec_scle,groupby='leiden',show_gene_labels=True,swap_axes=True,n_genes=20)

In [ ]:
Cell_Type2 = {'0': 'IFN vECs',
'1': 'CXCL9/10/11+ IFN vECs',
}

ec_scle.obs['Cell_Type2'] = ec_scle.obs['leiden'].map(Cell_Type2)

In [ ]:
ec_scle_map = ec_scle.obs['Cell_Type2']
overlapping_indices = ec.obs.index.intersection(ec_scle_map.index)

ec.obs['Cell_Type2'] = ec.obs['Cell_Type2'].astype(str)
ec.obs.loc[overlapping_indices, 'Cell_Type2'] = ec_scle_map.loc[overlapping_indices]

sc.pl.umap(ec, color="Cell_Type2")

In [ ]:
ec_map = ec.obs['Cell_Type2']
overlapping_indices = adata_filtered.obs.index.intersection(ec_map.index)

adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type1'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = ec_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color="Cell_Type2")

## Keratinocyte

In [ ]:
kera = adata_filtered[adata_filtered.obs["Cell_Type"]=="Keratinocytes"].copy()

sc.pp.pca(kera,random_state=42)
sc.pp.neighbors(kera,random_state=42)
sc.tl.umap(kera,random_state=42)

In [ ]:
sc.pl.umap(kera, color=["Lesion1","Sample"])

In [ ]:
sc.pl.umap(kera,color=["DMKN","CDSN","TMEM45A","DST","COL17A1","DSG1",
                       "DCT","MITF","MLANA",
                       "SOX9",])

In [ ]:
sc.tl.leiden(kera, resolution=0.3, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(kera,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(kera,groupby='leiden')
sc.tl.dendrogram(kera,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(kera,groupby='leiden',show_gene_labels=True,swap_axes=True, n_genes=15)

In [ ]:
Cell_Type1 = {'0': 'Injury-associated Keratinocytes',
 '1': 'Melanocytes',
 '2': 'Basal Cells',
 '3': 'Hair Follicle Epithelia',
 '4': 'Sebocytes',
 '5': 'Hair Follicle Epithelia',
 '6': 'Hair Follicle Epithelia',
 '7': 'IFN Keratinocytes',
 '8': 'Granular Keratinocytes',
 '9': 'Spinous Keratinocytes',
}
kera.obs['Cell_Type1'] = kera.obs['leiden'].map(Cell_Type1)

sc.pl.umap(kera,color=['Cell_Type1'])

In [ ]:
kera_map = kera.obs['Cell_Type1']
overlapping_indices = adata_filtered.obs.index.intersection(kera_map.index)

adata_filtered.obs['Cell_Type1'] = adata_filtered.obs['Cell_Type1'].astype(str)
adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type1'] = kera_map.loc[overlapping_indices]
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = kera_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color=["Cell_Type1","Cell_Type2"], ncols=1)

### IFN Keratinocytes

In [ ]:
kera1 = adata_filtered[adata_filtered.obs["Cell_Type1"]=="IFN Keratinocytes"].copy()

sc.pp.pca(kera1,random_state=42)
sc.pp.neighbors(kera1,random_state=42)
sc.tl.umap(kera1, random_state=42)

In [ ]:
sc.pl.umap(kera1, color=["Sample","Lesion1"])

In [ ]:
sc.tl.leiden(kera1, resolution=0.1, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(kera1,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(kera1,groupby='leiden')
sc.tl.dendrogram(kera1,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(kera1,groupby='leiden',show_gene_labels=True,swap_axes=True, n_genes=15)

In [ ]:
Cell_Type2 = {'0': 'IFN Basal Cells',
 '1': 'IFN Spinous Keratinocytes',
}
kera1.obs['Cell_Type2'] = kera1.obs['leiden'].map(Cell_Type2)

sc.pl.umap(kera1,color=['Cell_Type2'])

In [ ]:
kera1_map = kera1.obs['Cell_Type2']
overlapping_indices = adata_filtered.obs.index.intersection(kera1_map.index)

adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = kera1_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color=["Cell_Type2"], ncols=1)

#### IFN Basal

In [ ]:
ifnBasal_scle3 = kera1[(kera1.obs["Cell_Type2"]=="IFN Basal Cells")&
                       (kera1.obs["Sample"]=="SCLE1")].copy()
ifnBasal_scle4 = kera1[(kera1.obs["Cell_Type2"]=="IFN Basal Cells")&
                       (kera1.obs["Sample"]=="SCLE2")].copy()
ifnBasal_scle5 = kera1[(kera1.obs["Cell_Type2"]=="IFN Basal Cells")&
                       (kera1.obs["Sample"]=="SCLE3")].copy()

sc.pp.pca(ifnBasal_scle3,random_state=42)
sc.pp.neighbors(ifnBasal_scle3,random_state=42)
sc.tl.umap(ifnBasal_scle3, random_state=42)

sc.pp.pca(ifnBasal_scle4,random_state=42)
sc.pp.neighbors(ifnBasal_scle4,n_neighbors=30, random_state=42)
sc.tl.umap(ifnBasal_scle4, random_state=42)

sc.pp.pca(ifnBasal_scle5,random_state=42)
sc.pp.neighbors(ifnBasal_scle5,random_state=42)
sc.tl.umap(ifnBasal_scle5, random_state=42)

In [ ]:
sc.pl.umap(ifnBasal_scle3, color=["CXCL9","CXCL10","CXCL11"])

In [ ]:
sc.tl.leiden(ifnBasal_scle3, resolution=0.3, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(ifnBasal_scle3,color=['leiden'])

In [ ]:
Cell_Type2 = {'0': 'IFN Basal Cells',
              '1': 'IFN Basal Cells',
              '2': 'CXCL9/10/11+ IFN Basal Cells',
}
ifnBasal_scle3.obs['Cell_Type2'] = ifnBasal_scle3.obs['leiden'].map(Cell_Type2)

In [ ]:
sc.pl.umap(ifnBasal_scle4, color=["CXCL9","CXCL10","CXCL11"])

In [ ]:
sc.tl.leiden(ifnBasal_scle4, resolution=1.2, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(ifnBasal_scle4,color=['leiden'])

In [ ]:
Cell_Type2 = {'0': 'IFN Basal Cells',
              '1': 'IFN Basal Cells',
              '2': 'CXCL9/10/11+ IFN Basal Cells',
              '3': 'IFN Basal Cells',
              '4': 'CXCL9/10/11+ IFN Basal Cells',
              '5': 'IFN Basal Cells',
              '6': 'IFN Basal Cells',
              '7': 'CXCL9/10/11+ IFN Basal Cells',
}
ifnBasal_scle4.obs['Cell_Type2'] = ifnBasal_scle4.obs['leiden'].map(Cell_Type2)

In [ ]:
sc.pl.umap(ifnBasal_scle5, color=["CXCL9","CXCL10","CXCL11"])

In [ ]:
sc.tl.leiden(ifnBasal_scle5, resolution=0.3, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(ifnBasal_scle5,color=['leiden'])

In [ ]:
Cell_Type2 = {'0': 'IFN Basal Cells',
              '1': 'CXCL9/10/11+ IFN Basal Cells',
}
ifnBasal_scle5.obs['Cell_Type2'] = ifnBasal_scle5.obs['leiden'].map(Cell_Type2)

In [ ]:
ifnBasal_scle3_map = ifnBasal_scle3.obs['Cell_Type2']
overlapping_indices = adata_filtered.obs.index.intersection(ifnBasal_scle3_map.index)
adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = ifnBasal_scle3_map.loc[overlapping_indices]

ifnBasal_scle4_map = ifnBasal_scle4.obs['Cell_Type2']
overlapping_indices = adata_filtered.obs.index.intersection(ifnBasal_scle4_map.index)
adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = ifnBasal_scle4_map.loc[overlapping_indices]

ifnBasal_scle5_map = ifnBasal_scle5.obs['Cell_Type2']
overlapping_indices = adata_filtered.obs.index.intersection(ifnBasal_scle5_map.index)
adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = ifnBasal_scle5_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color=["Cell_Type2"], ncols=1)

## Immune

In [ ]:
immune = adata_filtered[adata_filtered.obs["Cell_Type"]=="Immune Cells"].copy()

sc.pp.pca(immune,random_state=42)
sc.pp.neighbors(immune,random_state=42)
sc.tl.umap(immune,random_state=42)

In [ ]:
sc.pl.umap(immune,color=['log1p_n_genes_by_counts', 'log1p_total_counts',
                        'n_genes_by_counts', 'total_counts','cell_area',
                        'nucleus_ratio'])

In [ ]:
sc.pl.umap(immune, color=["Sample","Lesion","Lesion1"])

In [ ]:
sc.pl.umap(immune, color=["CD3G","CD3E","CD4","CD8A","CD68","CLEC10A","CLEC9A"])

In [ ]:
sc.tl.leiden(immune, resolution=0.2, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(immune,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(immune,groupby='leiden')
sc.tl.dendrogram(immune,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(immune,groupby='leiden',show_gene_labels=True,swap_axes=True)

In [ ]:
Cell_Type1 = {'0': 'Myeloid Cells',
 '1': 'Lymphoid Cells',
 '2': 'Myeloid Cells',
}
immune.obs['Cell_Type1'] = immune.obs['leiden'].map(Cell_Type1)

sc.pl.umap(immune,color=['Cell_Type1'])

In [ ]:
immune_map = immune.obs['Cell_Type1']
overlapping_indices = adata_filtered.obs.index.intersection(immune_map.index)

adata_filtered.obs['Cell_Type1'] = adata_filtered.obs['Cell_Type1'].astype(str)
adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type1'] = immune_map.loc[overlapping_indices]
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = immune_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color=["Cell_Type1","Cell_Type2"], ncols=1)

### Lymphoid

In [ ]:
lymphoid = adata_filtered[adata_filtered.obs["Cell_Type1"]=="Lymphoid Cells"].copy()

sc.pp.pca(lymphoid,random_state=42)
sc.pp.neighbors(lymphoid,random_state=42,)
sc.tl.umap(lymphoid,random_state=42)

In [ ]:
sc.pl.umap(lymphoid,color=['Sample',"Lesion","Lesion1"])

In [ ]:
sc.pl.umap(lymphoid,color=['CD3E',"CD3G","CD4","CD8A","CD8B", "CXCR3"])

In [ ]:
sc.tl.leiden(lymphoid, resolution=0.4, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(lymphoid,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(lymphoid,groupby='leiden')
sc.tl.dendrogram(lymphoid,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(lymphoid,groupby='leiden',show_gene_labels=True,swap_axes=True, n_genes=20)

In [ ]:
Cell_Type2 = {'0': 'T Cells',
 '1': 'T Cells',
 '2': 'IFN T Cells',
}
lymphoid.obs['Cell_Type2'] = lymphoid.obs['leiden'].map(Cell_Type2)

sc.pl.umap(lymphoid,color="Cell_Type2")

In [ ]:
lymphoid_map = lymphoid.obs['Cell_Type2']
overlapping_indices = adata_filtered.obs.index.intersection(lymphoid_map.index)

adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)

adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = lymphoid_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color=["Cell_Type2"], ncols=1)

#### T Cell

In [ ]:
tcell = lymphoid[lymphoid.obs["Cell_Type2"]=="T Cells"].copy()

sc.pp.pca(tcell,random_state=42)
sc.pp.neighbors(tcell,random_state=42)
sc.tl.umap(tcell,random_state=42)

In [ ]:
sc.pl.umap(tcell, color=["CD3E","CD4","CD8A","CD8B","CTLA4","FOXP3","TIGIT","CD19","MS4A1","TRDC"])

In [ ]:
sc.tl.leiden(tcell, resolution=0.4, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(tcell,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(tcell,groupby='leiden')
sc.tl.dendrogram(tcell,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(tcell,groupby='leiden',show_gene_labels=True,swap_axes=True)

In [ ]:
Cell_Type3 = {'0': 'CD4 T Cells',
'1': 'CD8 T Cells',
'2': 'CD4 T Cells',
'3': 'Myocytes',
}
tcell.obs['Cell_Type3'] = tcell.obs['leiden'].map(Cell_Type3)

sc.pl.umap(tcell,color="Cell_Type3")

In [ ]:
tcell_map = tcell.obs['Cell_Type3']
overlapping_indices = adata_filtered.obs.index.intersection(tcell_map.index)

adata_filtered.obs['Cell_Type3'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type3'] = tcell_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color="Cell_Type3")

##### CD4

In [ ]:
cd4 = tcell[tcell.obs["Cell_Type3"]=="CD4 T Cells"].copy()

sc.pp.pca(cd4,random_state=42)
sc.pp.neighbors(cd4,random_state=42)
sc.tl.umap(cd4,random_state=42)

In [ ]:
sc.pl.umap(cd4, color=["CD3E","CD4","CD8A","CD8B","CTLA4","FOXP3","CD19","MS4A1","TRDC"])

In [ ]:
sc.tl.leiden(cd4, resolution=1, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(cd4,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(cd4,groupby='leiden')
sc.tl.dendrogram(cd4,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(cd4,groupby='leiden',show_gene_labels=True,swap_axes=True)

In [ ]:
Cell_Type3 = {'0': 'Tregs',
 '1': 'CD4 T Cells',
 '2': 'CD4 T Cells',
 '3': 'CD4 T Cells',
 '4': 'CD4 T Cells',
 '5': 'IFN CD4 T Cells',
 '6': 'CD4 T Cells',
 '7': "\u03B3\u03B4 T Cells",
 '8': 'CD4 T Cells',
}
cd4.obs['Cell_Type3'] = cd4.obs['leiden'].map(Cell_Type3)

sc.pl.umap(cd4,color="Cell_Type3")

In [ ]:
cd4_map = cd4.obs['Cell_Type3']
overlapping_indices = adata_filtered.obs.index.intersection(cd4_map.index)

adata_filtered.obs['Cell_Type3'] = adata_filtered.obs['Cell_Type3'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type3'] = cd4_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color="Cell_Type3")

##### IFN T Cells

In [ ]:
ifnTcell = lymphoid[lymphoid.obs["Cell_Type2"]=="IFN T Cells"].copy()

sc.pp.pca(ifnTcell,random_state=42)
sc.pp.neighbors(ifnTcell,random_state=42)
sc.tl.umap(ifnTcell,random_state=42)

In [ ]:
sc.pl.umap(ifnTcell,color=['CD8A'])

In [ ]:
sc.tl.leiden(ifnTcell, resolution=0.5, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(ifnTcell,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(ifnTcell,groupby='leiden')
sc.tl.dendrogram(ifnTcell,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(ifnTcell,groupby='leiden',show_gene_labels=True,swap_axes=True)

In [ ]:
Cell_Type3 = {'0': 'IFN CD4 T Cells',
'1': 'IFN CD4 T Cells',
'2': 'IFN CD8 T Cells',
}
ifnTcell.obs['Cell_Type3'] = ifnTcell.obs['leiden'].map(Cell_Type3)

sc.pl.umap(ifnTcell,color="Cell_Type3")

In [ ]:
ifnTcell_map = ifnTcell.obs['Cell_Type3']
overlapping_indices = adata_filtered.obs.index.intersection(ifnTcell_map.index)

adata_filtered.obs['Cell_Type3'] = adata_filtered.obs['Cell_Type3'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type3'] = ifnTcell_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color="Cell_Type3")

### Myeloid

In [ ]:
myeloid = adata_filtered[adata_filtered.obs["Cell_Type1"]=="Myeloid Cells"].copy()

sc.pp.pca(myeloid,random_state=42)
sc.pp.neighbors(myeloid,random_state=42)
sc.tl.umap(myeloid,random_state=42)

In [ ]:
sc.pl.umap(myeloid,color=['Sample',"Lesion","Lesion1"])

In [ ]:
sc.pl.umap(myeloid, color=['CLEC10A','CLEC9A',"CD14",'LAMP3','CD207',"CD163","CD68","CSF3R"])

In [ ]:
sc.tl.leiden(myeloid, resolution=0.7, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(myeloid,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(myeloid,groupby='leiden')
sc.tl.dendrogram(myeloid,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(myeloid,groupby='leiden',show_gene_labels=True,swap_axes=True,n_genes=20)

In [ ]:
Cell_Type2 = {'0': 'Macrophages',
 '1': 'Macrophages',
 '2': 'Neutrophils',
 '3': 'Macrophages',
 '4': 'cDC2s',
 '5': 'cDC2s',
 '6': 'cDC1s',
 '7': 'pDCs',
 '8': 'mregDCs',
 '9': 'Langerhans',
}
myeloid.obs['Cell_Type2'] = myeloid.obs['leiden'].map(Cell_Type2)

sc.pl.umap(myeloid,color="Cell_Type2")

In [ ]:
myeloid_map = myeloid.obs['Cell_Type2']
overlapping_indices = adata_filtered.obs.index.intersection(myeloid_map.index)

adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs['Cell_Type3'] = adata_filtered.obs['Cell_Type3'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = myeloid_map.loc[overlapping_indices]
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type3'] = myeloid_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color=["Cell_Type2","Cell_Type3"], ncols=1)

#### Macrophages

In [ ]:
mac = adata_filtered[adata_filtered.obs["Cell_Type2"]=="Macrophages"].copy()

sc.pp.pca(mac,random_state=42)
sc.pp.neighbors(mac,random_state=42)
sc.tl.umap(mac,random_state=42)

In [ ]:
sc.pl.umap(mac,color=['Sample',"Lesion","Lesion1"])

In [ ]:
sc.tl.leiden(mac, resolution=0.4, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(mac,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(mac,groupby='leiden')
sc.tl.dendrogram(mac,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(mac,groupby='leiden',show_gene_labels=True,swap_axes=True,n_genes=20)

In [ ]:
Cell_Type3 = {'0': 'Hypoxic Macrophages',
              '1': 'IFN Macrophages',
              '2': 'Tissue-resident Macrophages'
}
mac.obs['Cell_Type3'] = mac.obs['leiden'].map(Cell_Type3)

sc.pl.umap(mac,color="Cell_Type3")

In [ ]:
mac_map = mac.obs['Cell_Type3']
overlapping_indices = adata_filtered.obs.index.intersection(mac_map.index)

adata_filtered.obs['Cell_Type3'] = adata_filtered.obs['Cell_Type3'].astype(str)
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type3'] = mac_map.loc[overlapping_indices]

sc.pl.umap(adata_filtered, color=["Cell_Type3"], ncols=1)

## Fibroblasts

In [ ]:
fibro = adata_filtered[adata_filtered.obs["Cell_Type"]=="Fibroblasts"].copy()

sc.pp.pca(fibro,random_state=42)
sc.pp.neighbors(fibro,n_neighbors=10,random_state=42)
sc.tl.umap(fibro,random_state=42)

In [ ]:
sc.pl.umap(fibro,color=['Sample',"Lesion","Lesion1"])

In [ ]:
sc.tl.leiden(fibro, resolution=0.5, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)
sc.pl.umap(fibro,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(fibro,groupby='leiden')
sc.tl.dendrogram(fibro,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(fibro,groupby='leiden',show_gene_labels=True,swap_axes=True,n_genes=20)

In [ ]:
Cell_Type1 = {'0': 'ECM Remodeling Fibroblasts',
 '1': 'Schwann Cells',
 '2': 'IFN Fibroblasts',
 '3': 'Perivascular Fibroblasts',
 '4': 'Reticular Fibroblasts',
 '5': 'Reticular Fibroblasts',
 '6': 'Hair Follicle-associated Fibroblasts',
 '7': 'Papillary Fibroblasts',
}
fibro.obs['Cell_Type1'] = fibro.obs['leiden'].map(Cell_Type1)

sc.pl.umap(fibro,color="Cell_Type1")

In [ ]:
fibro_map = fibro.obs['Cell_Type1']
overlapping_indices = adata_filtered.obs.index.intersection(fibro_map.index)

adata_filtered.obs['Cell_Type1'] = adata_filtered.obs['Cell_Type1'].astype(str)
adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs['Cell_Type3'] = adata_filtered.obs['Cell_Type3'].astype(str)

adata_filtered.obs.loc[overlapping_indices, 'Cell_Type1'] = fibro_map.loc[overlapping_indices]
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = fibro_map.loc[overlapping_indices]
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type3'] = fibro_map.loc[overlapping_indices]

In [ ]:
ifnFibro = fibro[fibro.obs["Cell_Type1"]=="IFN Fibroblasts"].copy()

sc.pp.pca(ifnFibro,random_state=42)
sc.pp.neighbors(ifnFibro,n_neighbors=10,random_state=42)
sc.tl.umap(ifnFibro,random_state=42)

In [ ]:
sc.pl.umap(ifnFibro,color=['CXCL9',"CXCL10","CXCL11"])

In [ ]:
sc.tl.leiden(ifnFibro, resolution=1, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(ifnFibro,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(ifnFibro,groupby='leiden')
sc.tl.dendrogram(ifnFibro,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(ifnFibro,groupby='leiden',show_gene_labels=True,swap_axes=True,n_genes=15)

In [ ]:
Cell_Type2 = {'0': 'IFN Fibroblasts',
 '1': 'IFN Fibroblasts',
 '2': 'IFN Fibroblasts',
 '3': 'IFN Fibroblasts',
 '4': 'IFN Fibroblasts',
 '5': 'CXCL9/10/11+ IFN Fibroblasts',
 '6': 'CXCL9/10/11+ IFN Fibroblasts',
 '7': 'CXCL9/10/11+ IFN Fibroblasts',
 '8': 'IFN Fibroblasts',
}
ifnFibro.obs['Cell_Type2'] = ifnFibro.obs['leiden'].map(Cell_Type2)

sc.pl.umap(ifnFibro,color="Cell_Type2")

In [ ]:
ifnFibro_map = ifnFibro.obs['Cell_Type2']
overlapping_indices = adata_filtered.obs.index.intersection(ifnFibro_map.index)

adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs['Cell_Type3'] = adata_filtered.obs['Cell_Type3'].astype(str)

adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = ifnFibro_map.loc[overlapping_indices]
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type3'] = ifnFibro_map.loc[overlapping_indices]

In [ ]:
schwann = fibro[fibro.obs['Cell_Type1']=="Schwann Cells"].copy()

sc.pp.pca(schwann,random_state=42)
sc.pp.neighbors(schwann,random_state=42)
sc.tl.umap(schwann,random_state=42)

In [ ]:
sc.tl.leiden(schwann, resolution=0.3, neighbors_key='neighbors',random_state=42,flavor="igraph",n_iterations=2)

sc.pl.umap(schwann,color=['leiden'])

In [ ]:
sc.tl.rank_genes_groups(schwann,groupby='leiden')
sc.tl.dendrogram(schwann,groupby='leiden')
sc.pl.rank_genes_groups_heatmap(schwann,groupby='leiden',show_gene_labels=True,swap_axes=True,n_genes=20)

In [ ]:
Cell_Type1 = {'0': 'Schwann Cells',
 '1': 'Perineural Fibroblasts',
 '2': 'Schwann Cells',
}
schwann.obs['Cell_Type1'] = schwann.obs['leiden'].map(Cell_Type1)

sc.pl.umap(schwann,color="Cell_Type1")

In [ ]:
schwann_map = schwann.obs['Cell_Type1']
overlapping_indices = adata_filtered.obs.index.intersection(schwann_map.index)

adata_filtered.obs['Cell_Type1'] = adata_filtered.obs['Cell_Type1'].astype(str)
adata_filtered.obs['Cell_Type2'] = adata_filtered.obs['Cell_Type2'].astype(str)
adata_filtered.obs['Cell_Type3'] = adata_filtered.obs['Cell_Type3'].astype(str)

adata_filtered.obs.loc[overlapping_indices, 'Cell_Type1'] = schwann_map.loc[overlapping_indices]
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type2'] = schwann_map.loc[overlapping_indices]
adata_filtered.obs.loc[overlapping_indices, 'Cell_Type3'] = schwann_map.loc[overlapping_indices]

In [ ]:
sc.pl.umap(adata_filtered, color=["Cell_Type","Cell_Type1","Cell_Type2","Cell_Type3"], ncols=1)

In [ ]:
adata_filtered.write_h5ad("/Users/minhu/irAE1/Processed_Data/adata_filtered.h5ad")

# Save to Xenium Explorer

In [ ]:
SCLE_1_Cell_Type = SCLE_1.tables['table']
SCLE_2_Cell_Type = SCLE_2.tables['table']

SCLE_3_Cell_Type = SCLE_3.tables['table']
SCLE_4_Cell_Type = SCLE_4.tables['table']
SCLE_5_Cell_Type = SCLE_5.tables['table']

NS1_Cell_Type = NS1.tables['table']
NS2_Cell_Type = NS2.tables['table']
NS3_Cell_Type = NS3.tables['table']
NS4_Cell_Type = NS4.tables['table']

In [ ]:
Duplicates = adata_filtered.obs.columns[adata_filtered.obs.columns.isin(SCLE_1_Cell_Type.obs.columns)].to_list()
Duplicates.remove('cell_id')

In [ ]:
SCLE_1_Cell_Type.obs = pd.merge(SCLE_1_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='irAE-SCLE1'].obs.drop(columns=Duplicates),how='left',on='cell_id')
SCLE_2_Cell_Type.obs = pd.merge(SCLE_2_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='irAE-SCLE2'].obs.drop(columns=Duplicates),how='left',on='cell_id')

SCLE_3_Cell_Type.obs = pd.merge(SCLE_3_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='SCLE1'].obs.drop(columns=Duplicates),how='left',on='cell_id')
SCLE_4_Cell_Type.obs = pd.merge(SCLE_4_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='SCLE2'].obs.drop(columns=Duplicates),how='left',on='cell_id')
SCLE_5_Cell_Type.obs = pd.merge(SCLE_5_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='SCLE3'].obs.drop(columns=Duplicates),how='left',on='cell_id')

NS1_Cell_Type.obs = pd.merge(NS1_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='NS1'].obs.drop(columns=Duplicates),how='left',on='cell_id')
NS2_Cell_Type.obs = pd.merge(NS2_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='NS2'].obs.drop(columns=Duplicates),how='left',on='cell_id')
NS3_Cell_Type.obs = pd.merge(NS3_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='NS3'].obs.drop(columns=Duplicates),how='left',on='cell_id')
NS4_Cell_Type.obs = pd.merge(NS4_Cell_Type.obs,adata_filtered[adata_filtered.obs['Sample']=='NS4'].obs.drop(columns=Duplicates),how='left',on='cell_id')

In [ ]:
sopa.io.explorer.write_cell_categories('/Users/minhu/irAE/Data/SCLE/output-XETG00516__0076678__Region_1__20251017__213834/', SCLE_1_Cell_Type)
sopa.io.explorer.write_cell_categories('/Users/minhu/irAE/Data/SCLE/output-XETG00516__0076678__Region_2__20251017__213834/', SCLE_2_Cell_Type)

sopa.io.explorer.write_cell_categories('/Users/minhu/irAE1/Data/SCLE/output-XETG00516__0084920__Region_3__20260401__191409/', SCLE_3_Cell_Type)
sopa.io.explorer.write_cell_categories('/Users/minhu/irAE1/Data/SCLE/output-XETG00516__0084912__Region_4__20260401__191409/', SCLE_4_Cell_Type)
sopa.io.explorer.write_cell_categories('/Users/minhu/irAE1/Data/SCLE/output-XETG00516__0084920__Region_1__20260401__191409/', SCLE_5_Cell_Type)

sopa.io.explorer.write_cell_categories('/Users/minhu/irAE1/Data/Normal/output-XETG00516__0084912__Region_2__20260401__191409/', NS1_Cell_Type)
sopa.io.explorer.write_cell_categories('/Users/minhu/irAE1/Data/Normal/output-XETG00516__0084920__Region_2__20260401__191409/', NS2_Cell_Type)
sopa.io.explorer.write_cell_categories('/Users/minhu/irAE1/Data/Normal/output-XETG00516__0084912__Region_3__20260401__191409/', NS3_Cell_Type)
sopa.io.explorer.write_cell_categories('/Users/minhu/irAE1/Data/Normal/output-XETG00516__0084912__Region_5__20260401__191409/', NS4_Cell_Type)

# Load Processed Data

In [ ]:
adata_filtered = sc.read_h5ad("/Users/minhu/irAE1/Processed_Data/adata_filtered.h5ad")